# OpenMind — Cognitive Analytics Pipeline

This notebook analyzes a user's ChatGPT conversation history to surface hidden thinking patterns, behavioral trends, and personality insights.

**How it works:** We take the raw export JSON (the only thing we know is its structure), and through a series of ML, NLP, and statistical steps, we extract everything — topics, emotions, entities, interaction patterns, wellness signals — without making a single assumption about who the user is or what they discuss.

**Pipeline Overview:**
Upload → Parse → Preprocess → EDA → Embeddings → BERTopic → Emotion Model → Zero-Shot Classification → NER → Statistical Analysis → Compile → LLM Generation → Dashboard

---

## Step 1 — Environment Setup
Installing all required packages and configuring the runtime. This notebook is optimized for Google Colab with a T4 GPU.

In [1]:
%%capture
!pip install sentence-transformers bertopic umap-learn hdbscan plotly transformers torch langdetect ruptures openai

import warnings; warnings.filterwarnings("ignore")
import json, os, re, time
from datetime import datetime, timezone
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
print("Ready")

## Step 2 — Data Ingestion
ChatGPT exports follow a specific JSON structure where each conversation contains a `mapping` object with nested message nodes. Each node holds the message content (`content.parts`), author role (`author.role`), and timestamp (`create_time`). This step handles both single-file and multi-chunk exports.

In [ ]:
from google.colab import files
print("Upload ChatGPT export JSON file(s):")
uploaded = files.upload()

Upload ChatGPT export JSON file(s):


In [ ]:
all_conversations = []
for fname, content in uploaded.items():
    if not fname.endswith(".json"): continue
    try:
        data = json.loads(content)
        if isinstance(data, list):
            all_conversations.extend(data)
        elif isinstance(data, dict):
            for key in ["conversations","data","items","messages"]:
                if key in data and isinstance(data[key], list):
                    all_conversations.extend(data[key]); break
            else:
                all_conversations.append(data)
        print(f"  {fname}: OK")
    except Exception as e:
        print(f"  {fname}: FAILED ({e})")
print(f"\nConversations loaded: {len(all_conversations)}")

In [ ]:
# Walk through each conversation's mapping tree and extract user messages
records = []
for conv in all_conversations:
    cid = conv.get("conversation_id", conv.get("id",""))
    title = conv.get("title","Untitled") or "Untitled"
    ct = conv.get("create_time")
    mapping = conv.get("mapping",{})
    depth = 0
    for nid, node in mapping.items():
        msg = node.get("message")
        if not msg: continue
        role = msg.get("author",{}).get("role","unknown")
        depth += 1
        parts = msg.get("content",{}).get("parts",[])
        text_parts = []
        for p in parts:
            if isinstance(p, str): text_parts.append(p)
            elif isinstance(p, dict): text_parts.append(p.get("text",""))
        text = " ".join(text_parts).strip()
        if not text or len(text) < 3: continue
        ts = msg.get("create_time") or ct
        dt = None
        if ts:
            try: dt = datetime.fromtimestamp(float(ts), tz=timezone.utc)
            except: pass
        records.append({"cid":cid,"title":title,"role":role,"text":text,"timestamp":dt,"char_len":len(text)})
    for r in records:
        if r["cid"] == cid: r["conv_depth"] = depth

df_all = pd.DataFrame(records)
df = df_all[df_all["role"]=="user"].drop_duplicates(subset=["text","timestamp"]).reset_index(drop=True)
print(f"All messages: {len(df_all):,} | User messages: {len(df):,} | Conversations: {df['cid'].nunique():,}")

## Step 3 — Feature Engineering & Preprocessing
Before running any ML model, we extract structural and temporal features from each message. Language detection identifies the user's primary language. Text statistics (word count, sentence count, complexity) establish the baseline distribution. Temporal features (hour, day, month) enable behavioral pattern analysis later.

In [ ]:
# Detect primary language (supports 50+ languages via langdetect)
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def safe_detect(text):
    try: return detect(text[:500])
    except: return "unknown"

print("Detecting languages...")
df["language"] = df["text"].apply(safe_detect)
lang_dist = df["language"].value_counts()
primary_language = lang_dist.index[0]
print(f"Primary language: {primary_language} ({lang_dist.iloc[0]/len(df)*100:.1f}%)")
print(f"All languages: {lang_dist.head(10).to_dict()}")

# Compute text-level features for each message
df["word_count"] = df["text"].str.split().str.len()
df["sentence_count"] = df["text"].str.count(r"[.!?!]+") + 1
df["avg_word_len"] = df["text"].apply(lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0)
df["has_question"] = df["text"].str.contains(r"\?").astype(int)
df["has_code"] = df["text"].str.contains(r"```").astype(int)
df["has_url"] = df["text"].str.contains(r"https?://").astype(int)
df["complexity"] = (np.log1p(df["word_count"]) * df["avg_word_len"] * np.log1p(df["sentence_count"])).round(2)

# Extract temporal features for behavioral analysis
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["dow"] = df["timestamp"].dt.dayofweek
df["day_name"] = df["timestamp"].dt.day_name()
df["year_month"] = df["timestamp"].dt.strftime("%Y-%m")
df["year"] = df["timestamp"].dt.year
df["week"] = df["timestamp"].dt.to_period("W").astype(str)
df["is_late_night"] = ((df["hour"]>=23)|(df["hour"]<=4)).astype(int)
df["is_weekend"] = (df["dow"]>=5).astype(int)

print(f"\nFeatures: {len(df.columns)} columns")
print(f"Date range: {df['timestamp'].min():%Y-%m-%d} to {df['timestamp'].max():%Y-%m-%d}")

## Step 4 — Exploratory Data Analysis
Understanding the data before modeling. This step computes scale metrics, text distribution statistics, temporal patterns, conversation depth, and activity variability. Crucially, it also computes **adaptive parameters** — every threshold used downstream (short message cutoff, clustering sensitivity, outlier bounds) is derived from this user's actual data distribution, not hardcoded.

In [ ]:
print("="*60)
print("  EXPLORATORY DATA ANALYSIS")
print("="*60)

# Dataset scale
total_days = (df["timestamp"].max()-df["timestamp"].min()).days
months = sorted(df["year_month"].unique())
print(f"\n[SCALE]")
print(f"  Messages: {len(df):,} | Conversations: {df['cid'].nunique():,}")
print(f"  Span: {total_days} days ({total_days/365:.1f} years) | Active months: {len(months)}")
print(f"  Total words: {df['word_count'].sum():,} (~{df['word_count'].sum()//250} book pages)")
print(f"  Language: {primary_language}")

# Message length distribution
print(f"\n[TEXT DISTRIBUTION]")
print(f"  Words/msg: min={df['word_count'].min()} med={df['word_count'].median():.0f} mean={df['word_count'].mean():.1f} max={df['word_count'].max()}")
print(f"  Skew: {df['word_count'].skew():.2f}")
print(f"  <10 words: {(df['word_count']<10).mean()*100:.1f}% | >100: {(df['word_count']>100).mean()*100:.1f}% | >500: {(df['word_count']>500).mean()*100:.1f}%")

# Temporal patterns
monthly_counts = df.groupby("year_month").size()
print(f"\n[TEMPORAL]")
print(f"  Monthly: min={monthly_counts.min()} med={monthly_counts.median():.0f} max={monthly_counts.max()}")
print(f"  Peak month: {monthly_counts.idxmax()} ({monthly_counts.max()} msgs)")
print(f"  Peak hour: {df['hour'].mode().iloc[0]}:00 UTC")
print(f"  Peak day: {df['day_name'].mode().iloc[0]}")
print(f"  Late night (11pm-4am): {df['is_late_night'].mean()*100:.1f}%")
print(f"  Weekend: {df['is_weekend'].mean()*100:.1f}%")

# Conversation engagement depth
cs = df.groupby("cid").size()
print(f"\n[CONVERSATIONS]")
print(f"  Msgs/conv: min={cs.min()} med={cs.median():.0f} mean={cs.mean():.1f} max={cs.max()}")
print(f"  Single-msg convos: {(cs==1).sum()} ({(cs==1).mean()*100:.1f}%)")
print(f"  Deep convos (20+): {(cs>=20).sum()} | Marathon (50+): {(cs>=50).sum()}")

# Weekly activity consistency
weekly = df.groupby("week").size()
cv = weekly.std()/max(weekly.mean(),1)
daily = df.groupby("date").size()
active_days = len(daily)
total_range = len(pd.date_range(df["timestamp"].min(),df["timestamp"].max(),freq="D"))
print(f"\n[ACTIVITY PATTERN]")
print(f"  Weekly CoV: {cv:.2f} ({'very bursty' if cv>1.5 else 'bursty' if cv>1 else 'moderate' if cv>0.5 else 'consistent'})")
print(f"  Active days: {active_days}/{total_range} ({active_days/total_range*100:.1f}%)")
print(f"  Max single day: {daily.max()} msgs")

# Structural content signals
signals = {
    "Questions": df["has_question"].mean(),
    "Code blocks": df["has_code"].mean(),
    "URLs": df["has_url"].mean(),
}
print(f"\n[CONTENT SIGNALS]")
for k,v in sorted(signals.items(), key=lambda x:-x[1]):
    print(f"  {k}: {v*100:.1f}%")

# Compute adaptive parameters from distributions (no hardcoded values)
MIN_CLUSTER = max(5, min(15, len(df)//200))  # Lower for more granular topics
print(f"\n[ADAPTIVE PARAMS]")
print(f"  HDBSCAN min_cluster_size: {MIN_CLUSTER}")
print(f"  Can do timeline analysis: {len(months)>=3}")
print(f"  Can do phase detection: {len(months)>=6}")

In [ ]:
# Distribution overview charts
fig = make_subplots(rows=2,cols=2,subplot_titles=("Monthly Volume","Hourly Distribution","Day of Week","Message Length (capped 500w)"))
mc = df.groupby("year_month").size().reset_index(name="n")
fig.add_trace(go.Bar(x=mc["year_month"],y=mc["n"],marker_color="#00e5ff"),row=1,col=1)
hc = df.groupby("hour").size().reset_index(name="n")
fig.add_trace(go.Bar(x=hc["hour"],y=hc["n"],marker_color="#ffb300"),row=1,col=2)
dc = df.groupby("day_name").size().reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).reset_index(name="n")
dc.columns=["d","n"]
fig.add_trace(go.Bar(x=dc["d"],y=dc["n"],marker_color="#a78bfa"),row=2,col=1)
fig.add_trace(go.Histogram(x=df["word_count"].clip(upper=500),nbinsx=50,marker_color="#00d9a3"),row=2,col=2)
fig.update_layout(height=550,template="plotly_dark",plot_bgcolor="#0a0a1c",paper_bgcolor="#0a0a1c",showlegend=False,title="EDA Overview")
fig.update_xaxes(tickangle=-45,row=1,col=1)
fig.show()

## Step 5 — Sentence Embeddings
We encode every message into a 384-dimensional vector using `all-MiniLM-L6-v2`, a sentence transformer trained on over 1 billion text pairs. These embeddings capture semantic meaning — messages about similar topics end up close together in vector space regardless of exact wording. This is the foundation for topic clustering, similarity search, and zero-shot propagation in later steps.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
MAX_CHARS = 1000
texts = df["text"].str[:MAX_CHARS].tolist()

print(f"Embedding {len(texts):,} messages on {device}...")
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=256 if device=="cuda" else 64, normalize_embeddings=True)
print(f"Shape: {embeddings.shape}")

## Step 6 — Topic Discovery (BERTopic)
Unsupervised topic modeling using BERTopic, which combines UMAP dimensionality reduction, HDBSCAN density-based clustering, and c-TF-IDF for topic representation. We run an **adaptive search** across 6 different configurations (varying clustering method and sensitivity), score each result, and automatically select the best one. This ensures robust topic discovery regardless of dataset size or content diversity.

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
import umap as umap_lib
import hdbscan as hdbscan_lib
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english",
    min_df=1, max_df=0.95,
    ngram_range=(1,2), max_features=5000)

n_cl = len(df)
TARGET_TOPICS = max(8, min(25, int(np.sqrt(n_cl)/3)))

configs = [
    {"method":"eom","mc":max(10,n_cl//200),"label":"eom-medium"},
    {"method":"eom","mc":max(8,n_cl//300),"label":"eom-fine"},
    {"method":"leaf","mc":max(10,n_cl//200),"label":"leaf-medium"},
    {"method":"leaf","mc":max(8,n_cl//300),"label":"leaf-fine"},
    {"method":"eom","mc":max(5,n_cl//500),"label":"eom-granular"},
    {"method":"leaf","mc":max(15,n_cl//150),"label":"leaf-broad"},
]

best_model=None; best_topics=None; best_score=-1; best_label=""
print(f"Adaptive BERTopic (target={TARGET_TOPICS}, n={n_cl}):")
print(f"{'Config':18s} | {'mc':>3s} | {'topics':>6s} | {'noise':>6s} | {'score':>5s}")
print("-"*55)

for cfg in configs:
    hm = hdbscan_lib.HDBSCAN(min_cluster_size=cfg["mc"],min_samples=max(2,cfg["mc"]//5),
        metric="euclidean",cluster_selection_method=cfg["method"],prediction_data=True)
    tm = BERTopic(embedding_model=embed_model,
        umap_model=umap_lib.UMAP(n_components=10,n_neighbors=15,min_dist=0.0,metric="cosine",random_state=42),
        hdbscan_model=hm, vectorizer_model=vectorizer,
        representation_model=KeyBERTInspired(), verbose=False, calculate_probabilities=True)
    t, p = tm.fit_transform(texts, embeddings)
    nt = len(set(t))-(1 if -1 in t else 0)
    noise = (np.array(t)==-1).mean()
    if nt < 2: score = 0
    else:
        ts = max(0, 1.0-abs(np.log(max(nt,1))-np.log(TARGET_TOPICS))/np.log(TARGET_TOPICS))
        ns = max(0, 1.0-noise*2)
        rb = 0.2 if 5<=nt<=25 else 0.0
        score = ts*0.5+ns*0.3+rb
    print(f"{cfg['label']:18s} | {cfg['mc']:3d} | {nt:6d} | {noise*100:5.1f}% | {score:.3f}")
    if score > best_score: best_score=score; best_model=tm; best_topics=list(t); best_label=cfg["label"]

print(f"\n-> Winner: {best_label} (score={best_score:.3f})")
topic_model = best_model; topics = best_topics
n_found = len(set(topics))-(1 if -1 in topics else 0)

if n_found > TARGET_TOPICS * 1.5:
    print(f"Reducing {n_found} -> ~{TARGET_TOPICS}...")
    topic_model.reduce_topics(texts, nr_topics=TARGET_TOPICS)
    topics = topic_model.topics_
    n_found = len(set(topics))-(1 if -1 in topics else 0)

if sum(1 for t in topics if t==-1)/len(topics) > 0.30 and n_found >= 3:
    print(f"Reducing outliers...")
    topics = topic_model.reduce_outliers(texts, topics, strategy="embeddings", threshold=0.35)
    topic_model.update_topics(texts, topics=topics, vectorizer_model=vectorizer)

df["topic_id"] = topics
# Initialize probability tracking for topic assignments zeros.
probs = None

if isinstance(probs, np.ndarray) and probs.ndim == 2:
    df["topic_prob"] = [p.max() if len(p)>0 else 0 for p in probs]
else:
    df["topic_prob"] = np.array(probs) if probs is not None else np.zeros(len(df))

topic_info = topic_model.get_topic_info()
print(f"\nFinal: {n_found} topics")
for _, row in topic_info.head(20).iterrows():
    if row["Topic"] == -1: continue
    print(f"  Topic {row['Topic']:3d} ({row['Count']:4d} msgs): {row['Name'][:60]}")

In [ ]:
topic_labels = {}
for _, row in topic_info.iterrows():
    tid = row["Topic"]
    if tid == -1:
        topic_labels[tid] = "Other"
    else:
        name = re.sub(r"^\d+_", "", str(row.get("Name",""))).replace("_"," ").strip()
        words = [w for w in name.split() if len(w) > 1 and not w.startswith("#")]
        topic_labels[tid] = " ".join(w.capitalize() for w in words[:5]) if words else f"Topic {tid}"

df["topic_label"] = df["topic_id"].map(topic_labels)

print("UMAP 2D projection...")
umap_2d = umap_lib.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42).fit_transform(embeddings)
df["umap_x"], df["umap_y"] = umap_2d[:,0], umap_2d[:,1]
n_topics = len([t for t in topic_labels if t != -1])
print(f"{n_topics} topics labeled")
for tid in sorted(topic_labels):
    if tid == -1: continue
    cnt = (df["topic_id"]==tid).sum()
    if cnt > 0: print(f"  [{cnt:4d}] {topic_labels[tid]}")

## Step 7 — Emotion Analysis
Each message is classified across 7 emotion categories (anger, disgust, fear, joy, neutral, sadness, surprise) using `j-hartmann/emotion-english-distilroberta-base`, a fine-tuned RoBERTa model. The model outputs probability scores per emotion, which we use to compute a compound sentiment value and track emotional trends over time.

In [ ]:
from transformers import pipeline

print("Loading emotion model...")
emotion_pipe = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base",
                        device=0 if device=="cuda" else -1, truncation=True, max_length=512, top_k=None)

print(f"Analyzing emotions...")
BATCH = 64
emotion_results = []
for i in range(0, len(df), BATCH):
    batch = df["text"].iloc[i:i+BATCH].str[:500].tolist()
    try:
        emotion_results.extend(emotion_pipe(batch))
    except:
        for t in batch:
            try: emotion_results.extend(emotion_pipe([t]))
            except: emotion_results.append([{"label":"neutral","score":1.0}])
    if (i//BATCH) % 15 == 0:
        print(f"  {min(i+BATCH,len(df))}/{len(df)}")

# Extract per-emotion probability scores
for i, r in enumerate(emotion_results):
    scores = {x["label"]:x["score"] for x in r}
    df.loc[i,"emotion"] = max(scores, key=scores.get)
    df.loc[i,"emotion_conf"] = max(scores.values())
    for emo in ["anger","disgust","fear","joy","neutral","sadness","surprise"]:
        df.loc[i,f"emo_{emo}"] = scores.get(emo, 0)

# Compute compound sentiment: positive emotions minus negative emotions
df["sentiment"] = df["emo_joy"] + df["emo_surprise"]*0.5 - df["emo_anger"] - df["emo_sadness"] - df["emo_fear"] - df["emo_disgust"]

print(f"\nEmotion distribution:")
print(df["emotion"].value_counts().to_string())
print(f"\nAvg compound sentiment: {df['sentiment'].mean():.3f}")

## Step 8 — Interaction Pattern Classification
This step identifies *how* the user interacts with AI — not what they discuss, but what they're doing: asking questions, giving instructions, brainstorming, seeking advice, analyzing, etc. We use a zero-shot classifier (`DeBERTa-v3-base-mnli`) on a representative sample, then propagate labels to all messages via embedding similarity for efficiency.

In [ ]:
from transformers import pipeline as hf_pipeline

print("Loading zero-shot model...")
zs_pipe = hf_pipeline("zero-shot-classification", model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
                       device=0 if device=="cuda" else -1)

# Interaction types — these describe HOW someone uses AI, not WHAT they discuss
UNIVERSAL_TYPES = [
    "asking a factual question",
    "requesting creative writing or content",
    "asking for code or technical help",
    "seeking advice or guidance",
    "making a plan or organizing",
    "analyzing or researching a topic",
    "emotional expression or venting",
    "casual conversation or chitchat",
    "giving instructions or commands",
    "learning or studying a new concept",
    "brainstorming or generating ideas",
    "decision making or comparing options",
    "professional or career related",
    "personal reflection or self-improvement",
]

# Classify a sample via zero-shot, then propagate to all messages via embedding similarity
SAMPLE_N = min(500, len(df))
np.random.seed(42)
sample_idx = np.random.choice(len(df), SAMPLE_N, replace=False)
sample_texts = df.iloc[sample_idx]["text"].str[:300].tolist()

print(f"Zero-shot classifying {SAMPLE_N} sample messages...")
zs_results = []
for i, text in enumerate(sample_texts):
    try:
        r = zs_pipe(text, UNIVERSAL_TYPES, multi_label=False)
        zs_results.append({"type": r["labels"][0], "conf": r["scores"][0]})
    except:
        zs_results.append({"type": "casual conversation or chitchat", "conf": 0.5})
    if i % 50 == 0:
        print(f"  {i}/{SAMPLE_N}")

# Build prototype vectors for each discovered interaction type
sample_types = [r["type"] for r in zs_results]
type_embeddings = {}
for t in set(sample_types):
    idxs = [sample_idx[i] for i, r in enumerate(zs_results) if r["type"] == t]
    if idxs:
        type_embeddings[t] = embeddings[idxs].mean(axis=0)

# Assign types to all messages using cosine similarity to prototypes
print("Propagating to all messages via embedding similarity...")
type_names = list(type_embeddings.keys())
type_vecs = np.array([type_embeddings[t] for t in type_names])
type_vecs = type_vecs / np.linalg.norm(type_vecs, axis=1, keepdims=True)

sims = embeddings @ type_vecs.T
df["msg_type"] = [type_names[i] for i in sims.argmax(axis=1)]
df["msg_type_conf"] = sims.max(axis=1)

print(f"\nMessage type distribution:")
for t, c in df["msg_type"].value_counts().items():
    print(f"  {t:45s} {c:5d} ({c/len(df)*100:.1f}%)")

## Step 9 — Named Entity Recognition
Automatically extracts people, organizations, locations, and other entities mentioned in conversations using `bert-base-NER`. No predefined keyword lists — the model detects entities purely from context. Post-processing merges subword fragments and filters noise.

In [ ]:
from transformers import pipeline as hf_pipeline2

print("Loading NER model...")
ner_pipe = hf_pipeline2("ner", model="dslim/bert-base-NER", aggregation_strategy="simple",
                         device=0 if device=="cuda" else -1)

print("Extracting entities...")
all_entities = defaultdict(list)
entity_per_msg = []

for i in range(0, len(df), BATCH):
    batch = df["text"].iloc[i:i+BATCH].str[:500].tolist()
    for j, text in enumerate(batch):
        try:
            ents = ner_pipe(text)
            msg_ents = []
            for e in ents:
                if e["score"] > 0.7:
                    etype = e["entity_group"]  # PER, ORG, LOC, MISC
                    eword = e["word"].strip()
                    if len(eword) > 1:
                        all_entities[etype].append(eword)
                        msg_ents.append(f"{etype}:{eword}")
            entity_per_msg.append(msg_ents)
        except:
            entity_per_msg.append([])
    if (i//BATCH) % 15 == 0:
        print(f"  {min(i+BATCH,len(df))}/{len(df)}")

df["entities"] = entity_per_msg

# Aggregate and deduplicate extracted entities
print(f"\nEntities found:")
for etype in ["PER","ORG","LOC","MISC"]:
    counter = Counter(all_entities.get(etype,[]))
    top = counter.most_common(10)
    if top:
        print(f"  {etype}: {', '.join([f'{w}({c})' for w,c in top])}")

# Keep top entities per category
entity_summary = {}
for etype in ["PER","ORG","LOC","MISC"]:
    entity_summary[etype] = Counter(all_entities.get(etype,[])).most_common(20)

## Step 10 — Statistical & Behavioral Analysis
Pure statistics — no ML models. This step runs change point detection (PELT algorithm) on weekly activity to find phase boundaries, computes activity burst z-scores, fits a linear regression on prompt complexity over time, measures vocabulary richness (Type-Token Ratio), and analyzes conversation depth patterns.

In [ ]:
import ruptures

# Change point detection — find statistically significant shifts in activity
weekly_ts = df.groupby("week").size().values
if len(weekly_ts) > 10:
    print("Change point detection (PELT)...")
    algo = ruptures.Pelt(model="rbf", min_size=4).fit(weekly_ts.reshape(-1,1))
    change_points = algo.predict(pen=10)
    weeks_list = sorted(df["week"].unique())
    phase_boundaries = []
    for cp in change_points[:-1]:  # last one is always end
        if cp < len(weeks_list):
            phase_boundaries.append(weeks_list[cp])
    print(f"  Phase boundaries detected at weeks: {phase_boundaries}")
else:
    phase_boundaries = []
    print("Not enough data for change point detection")

# Activity burst detection using z-scores
weekly_series = df.groupby("week").size()
z_scores = (weekly_series - weekly_series.mean()) / max(weekly_series.std(), 1)
bursts = z_scores[z_scores > 1.5].sort_values(ascending=False)
print(f"\nActivity bursts (z>1.5): {len(bursts)}")
for w, z in bursts.head(5).items():
    print(f"  {w}: z={z:.1f} ({weekly_series[w]} msgs)")

# Prompt complexity trend over time (linear regression)
from scipy import stats as scipy_stats
monthly_complexity = df.groupby("year_month")["complexity"].mean()
if len(monthly_complexity) > 3:
    x = np.arange(len(monthly_complexity))
    slope, intercept, r, p, se = scipy_stats.linregress(x, monthly_complexity.values)
    complexity_trend = {"slope": round(slope, 3), "r_squared": round(r**2, 3), "p_value": round(p, 4),
                        "direction": "increasing" if slope > 0 else "decreasing",
                        "significant": p < 0.05}
    print(f"\nComplexity trend: {complexity_trend['direction']} (slope={slope:.3f}, R2={r**2:.3f}, p={p:.4f})")
else:
    complexity_trend = {"direction": "insufficient data"}

# Vocabulary richness (Type-Token Ratio per month)
def ttr(texts):
    words = " ".join(texts).lower().split()
    if not words: return 0
    return len(set(words)) / len(words)

monthly_ttr = {}
for ym in months:
    m_texts = df[df["year_month"]==ym]["text"].tolist()
    monthly_ttr[ym] = round(ttr(m_texts), 4)
print(f"\nVocabulary richness (TTR): {np.mean(list(monthly_ttr.values())):.3f} avg")

# Conversation depth trend
monthly_depth = df.groupby("year_month").agg(
    avg_depth=("conv_depth","mean"),
    max_depth=("conv_depth","max"),
).round(1)

# Topic diversity per month
monthly_topic_diversity = df[df["topic_id"]!=-1].groupby("year_month")["topic_id"].nunique()

print(f"\nMonthly topic diversity: {monthly_topic_diversity.mean():.1f} avg unique topics/month")

In [ ]:
# Monthly phase detection — combines topic and interaction type signals
phase_data = []
for ym in months:
    m = df[df["year_month"]==ym]
    if len(m) < 3: continue

    # Dominant topic for this month
    valid_topics = m[m["topic_id"]!=-1]["topic_label"]
    top_topic = valid_topics.mode().iloc[0] if len(valid_topics) > 0 else "mixed"
    topic_diversity = valid_topics.nunique() if len(valid_topics) > 0 else 0

    # Dominant interaction type
    top_type = m["msg_type"].mode().iloc[0] if len(m) > 0 else "mixed"

    # Dominant non-neutral emotion
    dom_emotion = m["emotion"].mode().iloc[0] if len(m) > 0 else "neutral"

    phase_data.append({
        "month": ym,
        "msgs": len(m),
        "top_topic": top_topic,
        "topic_diversity": int(topic_diversity),
        "top_type": top_type,
        "dom_emotion": dom_emotion,
        "avg_sentiment": round(m["sentiment"].mean(), 3),
        "complexity": round(m["complexity"].mean(), 1),
    })

phase_df = pd.DataFrame(phase_data)
print("Monthly phase data:")
for _, row in phase_df.iterrows():
    print(f"  {row['month']}: {row['msgs']:3d} msgs | topic=\'{row['top_topic'][:30]}\' | type=\'{row['top_type'][:30]}\' | mood={row['dom_emotion']}")

## Step 11 — Compile Findings
All discovered patterns, metrics, and distributions are assembled into a single structured JSON object. This is the complete analytical output of the pipeline — ready to be consumed by the LLM for narrative generation and by the dashboard template for visualization.

In [ ]:
# Assemble all pipeline outputs into a single findings object
findings = {}

# Core statistics
findings["stats"] = {
    "total_messages": int(len(df)),
    "total_conversations": int(df["cid"].nunique()),
    "total_words": int(df["word_count"].sum()),
    "book_pages": int(df["word_count"].sum()//250),
    "date_start": df["timestamp"].min().strftime("%b %Y"),
    "date_end": df["timestamp"].max().strftime("%b %Y"),
    "span_days": int(total_days),
    "active_months": len(months),
    "primary_language": primary_language,
    "peak_hour_utc": int(df["hour"].mode().iloc[0]),
    "peak_day": df["day_name"].mode().iloc[0],
    "avg_words_per_msg": round(df["word_count"].mean(), 1),
    "late_night_pct": round(df["is_late_night"].mean()*100, 1),
    "weekend_pct": round(df["is_weekend"].mean()*100, 1),
    "question_pct": round(df["has_question"].mean()*100, 1),
    "code_pct": round(df["has_code"].mean()*100, 1),
    "weekly_coefficient_of_variation": round(cv, 2),
    "usage_pattern": "very bursty" if cv>1.5 else "bursty" if cv>1 else "moderate" if cv>0.5 else "consistent",
}

# Discovered topics (unsupervised)
findings["topics"] = []
for _, row in topic_info.iterrows():
    if row["Topic"] == -1: continue
    findings["topics"].append({
        "id": int(row["Topic"]),
        "label": topic_labels.get(row["Topic"], f"Topic {row['Topic']}"),
        "count": int(row["Count"]),
        "pct": round(row["Count"]/len(df)*100, 1),
    })

# Monthly topic evolution
topic_evo = []
for ym in months:
    m = df[(df["year_month"]==ym)&(df["topic_id"]!=-1)]
    if len(m) == 0: continue
    top3 = m["topic_label"].value_counts().head(3)
    topic_evo.append({"month":ym, "topics":{k:int(v) for k,v in top3.items()}})
findings["topic_evolution"] = topic_evo

# Interaction type distribution
findings["message_types"] = df["msg_type"].value_counts().to_dict()
findings["message_types"] = {k:int(v) for k,v in findings["message_types"].items()}

# Emotion distribution
findings["emotions"] = {
    "distribution": {k:int(v) for k,v in df["emotion"].value_counts().items()},
    "avg_sentiment": round(df["sentiment"].mean(), 3),
    "joy_pct": round((df["emotion"]=="joy").mean()*100, 1),
    "anger_pct": round((df["emotion"]=="anger").mean()*100, 1),
    "sadness_pct": round((df["emotion"]=="sadness").mean()*100, 1),
    "fear_pct": round((df["emotion"]=="fear").mean()*100, 1),
    "surprise_pct": round((df["emotion"]=="surprise").mean()*100, 1),
    "neutral_pct": round((df["emotion"]=="neutral").mean()*100, 1),
}

# Monthly emotional trend
emo_trend = []
for ym in months:
    m = df[df["year_month"]==ym]
    if len(m)==0: continue
    emo_trend.append({
        "month": ym,
        "msgs": len(m),
        "sentiment": round(m["sentiment"].mean(),3),
        "joy": round((m["emotion"]=="joy").mean(),3),
        "anger": round((m["emotion"]=="anger").mean(),3),
        "sadness": round((m["emotion"]=="sadness").mean(),3),
        "neutral": round((m["emotion"]=="neutral").mean(),3),
        "late_night_pct": round(m["is_late_night"].mean(),3),
    })
findings["emotional_trend"] = emo_trend

# Named entities
findings["entities"] = {}
for etype in ["PER","ORG","LOC","MISC"]:
    findings["entities"][etype] = [[{"name":n,"count":c} for n,c in entity_summary.get(etype,[])]]

# Behavioral metrics
findings["behavior"] = {
    "complexity_trend": {
        "slope": float(complexity_trend["slope"]),
        "r_squared": float(complexity_trend["r_squared"]),
        "p_value": float(complexity_trend["p_value"]),
        "direction": complexity_trend["direction"],
        "significant": bool(complexity_trend["significant"])
    },
    "avg_vocabulary_richness": round(np.mean(list(monthly_ttr.values())), 3),
    "activity_bursts": [[{"week":str(w),"z_score":round(float(z),1),"msgs":int(weekly_series[w])} for w,z in bursts.head(10).items()]],
    "phase_boundaries": [str(b) for b in phase_boundaries],
    "monthly_phases": phase_data,
    "avg_conversation_depth": round(df["conv_depth"].mean(), 1),
    "deep_conversations": int((cs>=20).sum()),
    "marathon_conversations": int((cs>=50).sum()),
}

# Most active conversations
findings["top_conversations"] = [
    {"title":row.name[1][:80],"msgs":int(row["msgs"])}
    for _,row in df.groupby(["cid","title"]).agg(msgs=("text","count")).sort_values("msgs",ascending=False).head(15).iterrows()
]

# Monthly volume
findings["monthly_volume"] = [[{"month":ym,"msgs":int(c)} for ym,c in monthly_counts.items()]]

# Heatmap data
heatmap = df.groupby(["dow","hour"]).size().reset_index(name="n")
findings["heatmap"] = heatmap.to_dict(orient="records")

# Vocabulary richness trend
findings["vocabulary_trend"] = [[{"month":k,"ttr":v} for k,v in monthly_ttr.items()]]

print(f"Findings compiled: {len(json.dumps(findings))//1024} KB")
print(f"Keys: {list(findings.keys())}")

## Step 12 — Visualizations
Interactive Plotly charts rendered directly in the notebook for verification. These help validate the pipeline output before generating the final dashboard.

In [ ]:
# Topic cluster visualization (UMAP 2D projection)
c = df[df["topic_id"]!=-1]
if len(c) > 0:
    fig = px.scatter(c, x="umap_x", y="umap_y", color="topic_label", hover_data=["text","msg_type","emotion"],
                     title="Topic Cluster Map (BERTopic: UMAP + HDBSCAN + c-TF-IDF)", opacity=0.6, height=700)
    fig.update_traces(marker=dict(size=4))
    fig.update_layout(template="plotly_dark",plot_bgcolor="#0a0a1c",paper_bgcolor="#0a0a1c",
                      xaxis=dict(showgrid=False,showticklabels=False,title=""),yaxis=dict(showgrid=False,showticklabels=False,title=""))
    fig.show()

In [ ]:
# Temporal trends: volume, sentiment, late-night usage
fig = make_subplots(rows=3,cols=1,subplot_titles=("Monthly Volume","Sentiment Trend","Late Night %"),shared_xaxes=True,vertical_spacing=0.08)
et = pd.DataFrame(findings["emotional_trend"])
fig.add_trace(go.Bar(x=et["month"],y=et["msgs"],marker_color="#00e5ff"),row=1,col=1)
fig.add_trace(go.Scatter(x=et["month"],y=et["sentiment"],mode="lines+markers",line=dict(color="#00d9a3",width=2)),row=2,col=1)
fig.add_trace(go.Bar(x=et["month"],y=et["late_night_pct"]*100,marker_color="#a78bfa"),row=3,col=1)
fig.update_layout(height=650,template="plotly_dark",plot_bgcolor="#0a0a1c",paper_bgcolor="#0a0a1c",showlegend=False)
fig.show()

In [ ]:
# Activity heatmap (day of week x hour)
hp = df.groupby(["day_name","hour"]).size().reset_index(name="n")
pv = hp.pivot_table(index="day_name",columns="hour",values="n",fill_value=0)
pv = pv.reindex([d for d in ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"] if d in pv.index])
fig = go.Figure(data=go.Heatmap(z=pv.values,x=[f"{h:02d}:00" for h in pv.columns],y=pv.index,colorscale="Viridis"))
fig.update_layout(title="Peak Activity Hours",height=350,template="plotly_dark",plot_bgcolor="#0a0a1c",paper_bgcolor="#0a0a1c")
fig.show()

In [ ]:
# Interaction types and emotion distributions
fig = make_subplots(rows=1,cols=2,subplot_titles=("Message Types (Zero-Shot)","Emotion Distribution"))
mt = df["msg_type"].value_counts().head(12).reset_index(); mt.columns=["type","n"]
fig.add_trace(go.Bar(y=mt["type"],x=mt["n"],orientation="h",marker_color="#00e5ff"),row=1,col=1)
ed = df["emotion"].value_counts().reset_index(); ed.columns=["emotion","n"]
fig.add_trace(go.Bar(y=ed["emotion"],x=ed["n"],orientation="h",marker_color="#ffb300"),row=1,col=2)
fig.update_layout(height=400,template="plotly_dark",plot_bgcolor="#0a0a1c",paper_bgcolor="#0a0a1c",showlegend=False)
fig.update_yaxes(autorange="reversed")
fig.show()

In [ ]:
# OpenAI API key (used for topic labeling and content generation)
OPENAI_API_KEY = ""  # @param {type:"string"}

if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("Enter your OpenAI API key: ")

## Step 12b — Data Completion & Validation
This step computes derived metrics that depend on multiple upstream outputs: wellness scoring (burnout factors, positivity ratio, cognitive load), topic co-occurrence edges for the brain map, abandoned topic detection, and monthly phase assignments. Every wellness metric uses simple linear scaling — no arbitrary multipliers. Observations are generated only when statistical tests show significance (p < 0.05).

In [ ]:
# ═══════════════════════════════════════
# Compute derived metrics and validate all required fields
# ═══════════════════════════════════════
from scipy import stats as sp_stats
from collections import defaultdict

N = len(df)
months_list = sorted(df["year_month"].unique())
daily_counts = df.groupby("date").size()
weekly_counts = df.groupby("week").size()
monthly_counts = df.groupby("year_month").size()
cv = weekly_counts.std() / max(weekly_counts.mean(), 1)

# ── Monthly aggregation ──
if "monthly" not in findings or not findings["monthly"]:
    src = findings.get("monthly_volume", [])
    if src:
        # Access nested list structure correctly
        findings["monthly"] = [[{"m":x_item["month"],"n":x_item["msgs"],"w":round(df[df["year_month"]==x_item["month"]]["word_count"].mean(),1)} for x_item in x] for x in src][0]
    else:
        findings["monthly"] = [[{"m":ym,"n":int(c),"w":round(df[df["year_month"]==ym]["word_count"].mean(),1)} for ym,c in monthly_counts.items()]][0]
    print(f"monthly: {len(findings['monthly'])}")

# ── Monthly phase assignments ──
if not findings.get("monthly_phases"):
    mp = []
    for ym in months_list:
        m = df[df["year_month"]==ym]
        if len(m)<3: continue
        v = m[m["topic_id"]!=-1]
        tt = v["topic_label"].mode().iloc[0] if len(v)>0 and len(v["topic_label"].mode())>0 else "mixed"
        mt = m["msg_type"].mode().iloc[0] if "msg_type" in m.columns and len(m["msg_type"].mode())>0 else max(findings.get("message_types",{"mixed":1}),key=findings.get("message_types",{"mixed":1}).get)
        nn = m[m["emotion"]!="neutral"] if "emotion" in m.columns else pd.DataFrame()
        de = nn["emotion"].mode().iloc[0] if len(nn)>0 and len(nn["emotion"].mode())>0 else "neutral"
        mp.append({"month":ym,"msgs":len(m),"top_topic":tt,"top_type":mt,"dom_emotion":de,
            "avg_sentiment":round(m["sentiment"].mean(),3) if "sentiment" in m.columns else 0,
            "complexity":round(m["complexity"].mean(),1) if "complexity" in m.columns else 0})
    findings["monthly_phases"] = mp
    print(f"monthly_phases: {len(mp)}")

# ── Wellness metrics (linear scaling) ──
pos_n = df["emotion"].isin(["joy","surprise"]).sum() if "emotion" in df.columns else 0
neg_n = df["emotion"].isin(["anger","sadness","fear","disgust"]).sum() if "emotion" in df.columns else 1
positivity_ratio = round(pos_n/max(neg_n,1),2)

avg_weekly = weekly_counts.mean()
late_pct = df["is_late_night"].mean()*100
non_neutral = df[df["emotion"]!="neutral"] if "emotion" in df.columns else pd.DataFrame()
neg_of_nn = neg_n/max(len(non_neutral),1)*100 if len(non_neutral)>10 else 30
cs = df.groupby("cid").size()
marathon_pct = (cs>=50).mean()*100

burnout_factors = {
    "Work Intensity": min(100, round(avg_weekly*2)),
    "Late Night": min(100, round(late_pct*3)),
    "Negative Sentiment": min(100, round(neg_of_nn)),
    "Volatility": min(100, round(cv*50)),
    "Session Depth": min(100, round(marathon_pct*10)),
}
burnout_score = round(sum(burnout_factors.values())/len(burnout_factors),1)

observations = []
if N > 30:
    late_obs = df["is_late_night"].sum()
    _,p_late = sp_stats.chisquare([late_obs,N-late_obs],[N*6/24,N*18/24])
    if p_late < 0.05 and late_obs > N*6/24:
        observations.append({"type":"attention","signal":"Late-Night Pattern","detail":f"{late_obs} msgs after 11pm ({late_pct:.0f}%, p={p_late:.4f})","suggestion":"Consider a digital wind-down routine."})
if "sentiment" in df.columns:
    ms = df.groupby("year_month")["sentiment"].mean()
    if len(ms) > 5:
        sl,_,_,ps,_ = sp_stats.linregress(np.arange(len(ms)),ms.values)
        if ps < 0.05:
            d = "declining" if sl<0 else "improving"
            observations.append({"type":"attention" if sl<0 else "positive","signal":f"Sentiment {d.title()}","detail":f"Significant trend (p={ps:.4f})","suggestion":f"Emotional tone is {d}."})
mc_z = (monthly_counts-monthly_counts.mean())/max(monthly_counts.std(),1)
for ym,z in mc_z[mc_z.abs()>2].head(3).items():
    observations.append({"type":"attention" if z>0 else "positive","signal":f"{'Surge' if z>0 else 'Quiet'}: {ym}","detail":f"{int(monthly_counts[ym])} msgs (z={z:.1f})","suggestion":"May indicate deadline pressure." if z>0 else "Rest is healthy."})
if positivity_ratio < 1.5 and neg_n > 10:
    observations.append({"type":"attention","signal":"Low Positivity","detail":f"Ratio: {positivity_ratio}","suggestion":"Negative emotions are relatively frequent."})
elif positivity_ratio > 4:
    observations.append({"type":"positive","signal":"Strong Positivity","detail":f"Ratio: {positivity_ratio}","suggestion":"Healthy emotional pattern."})

psych_monthly = []
for ym in months_list:
    m = df[df["year_month"]==ym]
    if len(m)==0: continue
    psych_monthly.append({"m":ym,"stress":int(m["emotion"].isin(["fear","anger"]).sum()) if "emotion" in m.columns else 0,
        "positive":int((m["emotion"]=="joy").sum()) if "emotion" in m.columns else 0,
        "frustration":int((m["emotion"]=="anger").sum()) if "emotion" in m.columns else 0,
        "late_pct":round(m["is_late_night"].mean(),3)})

findings["psych"] = {"positivity_ratio":positivity_ratio,"burnout_score":burnout_score,"burnout_factors":burnout_factors,
    "cognitive_load":{"avg_daily":round(daily_counts.mean(),1),"max_daily":int(daily_counts.max()),"std_daily":round(daily_counts.std(),1),"days_over_20":int((daily_counts>20).sum()),"days_over_50":int((daily_counts>50).sum())},
    "late_night_pct":round(late_pct,1),"night_msgs":int(df["is_late_night"].sum()),
    "sentiment_dist":{k:int(v) for k,v in df["emotion"].value_counts().items()} if "emotion" in df.columns else {},
    "observations":observations,"psych_monthly":psych_monthly}
print(f"psych: burnout={burnout_score}, obs={len(observations)}")

# ── Topic co-occurrence edges ──
ec = defaultdict(int)
for _,grp in df[df["topic_id"]!=-1].groupby("cid"):
    labels = grp["topic_label"].unique()
    for i in range(len(labels)):
        for j in range(i+1,len(labels)):
            ec[tuple(sorted([labels[i],labels[j]]))] += 1
findings["edges"] = [[{"source":p[0],"target":p[1],"weight":w} for p,w in sorted(ec.items(),key=lambda x:-x[1])[:25]]][0]
print(f"edges: {len(findings['edges'])}")

# ── Abandoned topic detection ──
recent = months_list[-6:] if len(months_list)>6 else months_list
old = [m for m in months_list if m not in recent]
abandoned = []
if old:
    ot = df[(df["year_month"].isin(old))&(df["topic_id"]!=-1)].groupby("topic_label").size()
    rt = df[(df["year_month"].isin(recent))&(df["topic_id"]!=-1)].groupby("topic_label").size()
    for t,c in ot.items():
        if t=="Other": continue
        if c>=8 and rt.get(t,0)==0:
            abandoned.append({"topic":t,"old_count":int(c),"first_seen":df[df["topic_label"]==t]["year_month"].min(),"last_seen":df[df["topic_label"]==t]["year_month"].max()})
    abandoned.sort(key=lambda x:-x["old_count"])
findings["abandoned_topics"] = abandoned[:10]
print(f"abandoned: {len(abandoned)}")

# ── NER fragment cleanup ──
if findings.get("entities"):
    for etype in findings["entities"]:
        cleaned = []
        # Unpack the nested list before iterating
        for item_list in findings["entities"][etype]:
            for item in item_list:
                name = re.sub(r"^#+\s*","",item["name"])
                name = re.sub(r"\s*#+\s*","",name).strip()
                if name.lower().startswith("ffworld"): name = "A" + name
                if len(name)<2: continue
                if name.lower() in {"the","a","an","is","it","this","that","i","my"}: continue
                cleaned.append({"name":name,"count":item["count"]})
        findings["entities"][etype] = cleaned
    print("NER cleaned")

# ── topic_evolution rebuild (with current labels) ──
te = []
for ym in months_list:
    m = df[(df["year_month"]==ym)&(df["topic_id"]!=-1)]
    if len(m)==0: continue
    top = m["topic_label"].value_counts().head(5)
    te.append({"month":ym,"topics":{k:int(v) for k,v in top.items()}})
findings["topic_evolution"] = te
print(f"topic_evolution rebuilt: {len(te)}")

# ── Verify ──
needed = ["stats","monthly","topics","topic_evolution","message_types","emotions","emotional_trend","entities","heatmap","edges","abandoned_topics","behavior","monthly_phases","top_conversations","vocabulary_trend","psych"]
miss = [k for k in needed if not findings.get(k) or (isinstance(findings[k],(list,dict)) and len(findings[k])==0)]
print(f"\nVerification: {len(needed)-len(miss)}/{len(needed)} present" + (f" MISSING: {miss}" if miss else " ALL OK"))

## Step 12c — LLM-Powered Topic Labeling
BERTopic discovers clusters accurately, but its TF-IDF labels are raw keywords (e.g., "Rent Bed Apartment Issues"). This step sends representative messages from each cluster to GPT-4o-mini, which reads the actual content and returns a meaningful 2-5 word name (e.g., "Hostel Management System"). This is the single highest-impact step for dashboard quality.

In [ ]:
import openai
client_label = openai.OpenAI(api_key=OPENAI_API_KEY)

top_topics = [t for t in findings["topics"] if t["count"] >= 10]
print(f"Labeling {len(top_topics)} topics...")

# Extract representative messages from each cluster for LLM labeling
topic_samples = {}
for t in top_topics:
    label = t["label"]
    msgs = df[df["topic_label"]==label]["text"].tolist()
    if len(msgs) <= 5: samples = msgs
    else:
        by_len = sorted(msgs, key=len)
        idx = np.linspace(0, len(by_len)-1, 5).astype(int)
        samples = [by_len[i] for i in idx]
    topic_samples[label] = [s[:200] for s in samples]

parts = [
    "Below are clusters from a ChatGPT history. Read the sample messages and give each a SHORT name (2-5 words).",
    "Be specific: 'Hostel Management System' not 'Web Development', 'MSc Dissertation Research' not 'Academic Work'.",
    "Return ONLY JSON: {\"old_label\": \"New Name\", ...}",
    ""
]
for label, samples in topic_samples.items():
    cnt = len(df[df["topic_label"]==label])
    parts.append(f"CLUSTER \"{label}\" ({cnt} msgs):")
    for s in samples: parts.append(f"  - {s}")
    parts.append("")

resp = client_label.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":"\n".join(parts)}],
    max_tokens=2000, temperature=0.3)
raw = resp.choices[0].message.content
raw = re.sub(r"^```json\n?","",raw.strip())
raw = re.sub(r"\n?```$","",raw.strip())

try:
    label_map = json.loads(raw)
    print(f"Renamed {len(label_map)} topics:")
    for old, new in label_map.items():
        cnt = len(df[df["topic_label"]==old])
        if cnt > 0:
            print(f"  [{cnt:4d}] {old[:30]:30s} -> {new}")
            df.loc[df["topic_label"]==old, "topic_label"] = new

    # Propagate new labels across all findings (evolution, phases, edges, abandoned)
    for t in findings["topics"]:
        if t["label"] in label_map: t["label"] = label_map[t["label"]]
    for te in findings.get("topic_evolution",[]):
        te["topics"] = {label_map.get(k,k):v for k,v in te.get("topics",{}).items()}
    for mp in findings.get("monthly_phases",[]):
        if mp.get("top_topic") in label_map: mp["top_topic"] = label_map[mp["top_topic"]]
    for e in findings.get("edges",[]):
        if e["source"] in label_map: e["source"] = label_map[e["source"]]
        if e["target"] in label_map: e["target"] = label_map[e["target"]]
    for a in findings.get("abandoned_topics",[]):
        if a["topic"] in label_map: a["topic"] = label_map[a["topic"]]
    print("All labels updated across findings")
except Exception as e:
    print(f"Failed: {e}. Raw: {raw[:300]}")

## Step 13 — Dynamic Content Generation
The LLM receives the complete findings JSON and generates all interpretive content: a unique personality archetype, journey narrative with chapters, behavioral insights, coaching recommendations, thinking style breakdown, cognitive biases, future path predictions, and a shareable card tagline. The dashboard template is pre-built — the LLM only fills the creative slots.

In [ ]:
import openai
client = openai.OpenAI(api_key=OPENAI_API_KEY)

findings_str = json.dumps(findings, indent=2, default=str)
if len(findings_str) > 80000:
    c = dict(findings)
    for key in ["topic_evolution","emotional_trend","monthly_volume","monthly_phases","vocabulary_trend","heatmap"]:
        if key in c and isinstance(c[key],list) and len(c[key])>24: c[key]=c[key][-24:]
    findings_str = json.dumps(c, indent=2, default=str)

print(f"Sending {len(findings_str)//1024}KB to GPT-4o...")

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role":"user","content":"""You are OpenMind, analyzing a users complete ChatGPT history. Read the data carefully — the topic labels, entities, message types, and monthly phases tell the story of a real person.

Generate a JSON object with EXACTLY these keys:

1. personality — object:
   - primary: UNIQUE archetype name specific to THIS person (not generic buzzwords)
   - description: 2-3 sentences referencing their actual topics, organizations, and trajectory
   - traits: 4-6 traits with percentages summing to 100. Trait names from their actual activities.

2. scores — array of 5 objects with name, value (0-100), description. Names from their real patterns.

3. thinking_style — array of 4-5 objects with name and pct (sum 100)

4. biases — array of 2-4 objects with name, severity (0-100), description

5. journey — object:
   - summary: one-line arc showing their evolution
   - chapters: array of 5-7 objects. IMPORTANT: Look at monthly_phases data spanning multiple years. Find AT LEAST 5 distinct phases. Each chapter: title (vivid, specific), period ("Jan - May 2023"), msgs (from data), narrative (one paragraph telling their story), color (hex).
   - Reference actual entities (organizations, technologies, locations) in narratives.

6. insights — array of 8-10 strings, each with SPECIFIC meaning based on numbers and findings from data and analysis

7. coaching — array of 4-6 objects with type (Action/Pattern/Habit), urgency (high/medium/low), text

8. future_paths — array of 3-5 objects with name, probability (sum ~100), reasoning

9. wellness_narrative — one paragraph about emotional and behavioral patterns

10. share_card — object with tagline and subtitle

Return ONLY valid JSON. No markdown. No backticks. No explanation.

ANALYSIS DATA:
""" + findings_str}],
    max_tokens=5000,
    temperature=0.7)

raw = response.choices[0].message.content
raw = re.sub(r"^```json\n?","",raw.strip())
raw = re.sub(r"\n?```$","",raw.strip())

try:
    dynamic = json.loads(raw)
    print(f"\nGenerated {len(dynamic)} keys:")
    if "personality" in dynamic:
        print(f"  Personality: {dynamic['personality'].get('primary')}")
        print(f"  Description: {dynamic['personality'].get('description','')[:100]}")
    if "journey" in dynamic:
        print(f"  Journey: {dynamic['journey'].get('summary')}")
        print(f"  Chapters: {len(dynamic['journey'].get('chapters',[]))}")
        for c in dynamic["journey"].get("chapters",[]):
            print(f"    {c.get('period','')}: {c.get('title','')}")
    if "scores" in dynamic:
        for s in dynamic["scores"]:
            print(f"  Score: {s.get('name')}: {s.get('value')}")
except json.JSONDecodeError as e:
    print(f"Parse error: {e}")
    m = re.search(r"\{[\s\S]*\}", raw)
    dynamic = json.loads(m.group()) if m else {}
    print(f"Extracted: {list(dynamic.keys())}" if dynamic else "FAILED - retry")

## Step 14 — Dashboard Assembly
The pre-built HTML template (with 3D character, brain map, interactive charts, and 10 navigable tabs) is embedded in this notebook as a base64 string. This step merges the pipeline findings and LLM-generated content into a single data payload, injects it into the template, and produces the final self-contained HTML file.

In [ ]:
import base64

merged = dict(findings)
if dynamic:
    merged.update(dynamic)
    print(f"Merged: pipeline + {len(dynamic)} LLM keys")
else:
    merged.update({"personality":{"primary":"Analysis Complete","description":"Retry Step 13.","traits":{}},"journey":{"summary":"","chapters":[]},"insights":[],"coaching":[],"future_paths":[],"scores":[],"thinking_style":[],"biases":[],"share_card":{"tagline":"","subtitle":""},"wellness_narrative":""})

data_json = json.dumps(merged, separators=(",",":"), default=str)
template = base64.b64decode("...").decode()
final = template.replace("__AI_MIRROR_DATA__", data_json)

with open("ai_mirror_dashboard.html","w") as f: f.write(final)
with open("ai_mirror_findings.json","w") as f: json.dump(findings,f,indent=2,default=str)
with open("ai_mirror_dynamic.json","w") as f: json.dump(dynamic or {},f,indent=2,default=str)
df.to_csv("ai_mirror_enriched.csv",index=False)
np.save("ai_mirror_embeddings.npy",embeddings)

print(f"\nDashboard: {len(final)//1024}KB")
print(f"Open ai_mirror_dashboard.html in browser")
display(HTML(final))
from google.colab import files
files.download("ai_mirror_dashboard.html")
files.download("ai_mirror_findings.json")

## Step 15 — Summary & Export

In [ ]:
print("=" * 60)
print("  OpenMind — COMPLETE")
print("=" * 60)
print(f"\n  {len(df):,} messages | {df['cid'].nunique()} conversations")
print(f"  {df['timestamp'].min():%Y-%m-%d} to {df['timestamp'].max():%Y-%m-%d}")
print(f"  {df['word_count'].sum():,} words (~{df['word_count'].sum()//250} book pages)")
print(f"\n  Topics: {len(findings.get('topics',[]))} discovered + LLM labeled")
print(f"  Types: {len(findings.get('message_types',{}))}")
print(f"  Entities: {sum(len(v) for v in findings.get('entities',{}).values())}")
print(f"  Burnout: {findings['psych']['burnout_score']}")
if dynamic:
    print(f"\n  Personality: {dynamic.get('personality',{}).get('primary','?')}")
    print(f"  Journey: {len(dynamic.get('journey',{}).get('chapters',[]))} chapters")
    print(f"  Arc: {dynamic.get('journey',{}).get('summary','?')}")
print(f"\n  Dashboard: ai_mirror_dashboard.html")
print("=" * 60)